# SQ2 — SEM with semopy

**Question:** How does perceived control over social media content differ across age groups?

This SEM follows the course example: describe the model, run it and check the results. It uses three survey items to measure a latent control factor, then relates that factor to age groups. This is called a MIMIC model. It links several age variables to one factor, measured by several questions. It does not prove cause and effect.

**Purpose:** practise SEM and check a candidate measurement model. The three questions are not strongly related. Keep the earlier SQ2 analysis as well.

**Steps:** load, code three items, add age-group indicators, fit SEM, check estimates.

Prepared with Codex assistance. Code structure adapted from the course `SEM.ipynb` and [semopy tutorial](https://semopy.com/tutorial.html). Document your actual individual contribution before submission.

## 1. Import semopy

Install the packages with `pip install -r requirements.txt` before running this notebook.

In [ ]:
from pathlib import Path
import pandas as pd
import semopy

print("semopy:", semopy.__version__)

## 2. Prepare the control items

- **c1:** I feel I am in control of my feed.
- **c2:** I can influence the type of content I get on my feed.
- **c3:** The algorithm decides what I watch (reverse coded).

All answers use 1 = Strongly disagree to 5 = Strongly agree. For c3, use **6 − score** so higher values mean less algorithmic control. This is a candidate item selection based on wording; the course thesis/codebook still needs checking before calling it a validated scale.

Leave out rows that are missing an answer needed for this model. Keep the source unchanged. Do not use ranks, min-max or z-scores as the input.

In [ ]:
# Load the source data.
raw = pd.read_excel(Path.home() / 'Downloads' / 'Responses (3).xlsx')
raw.columns = raw.columns.str.replace('\u00a0', ' ', regex=False).str.strip()
scale = {'Strongly disagree': 1, 'Disagree': 2, 'Neutral': 3,
         'Agree': 4, 'Strongly agree': 5}

# Select three possible control items.
items = {
    'c1': 'I feel I am in control of my feed',
    'c2': 'I can influence the type of content I get on my feed',
    'c3': 'The algorithm decides what I watch'
}
data = pd.DataFrame(index=raw.index)
for name, question in items.items():
    answers = raw[question].astype('string').str.strip()
    assert answers.dropna().isin(scale).all()
    data[name] = answers.map(scale).astype('Float64')

# Reverse the negative item and remove missing model answers.
data['c3'] = 6 - data['c3']
data['age'] = raw['What age group are you in?'].astype('string').str.strip()
data = data.dropna()
print('Included:', len(data), '| Excluded:', len(raw) - len(data))

## 3. Check the candidate scale

Cronbach’s alpha shows how closely the items are related. It does not prove that the items measure one factor or mean the same thing across ages. A weak result is a reason for caution. Do not remove items just to get a better result.

In [ ]:
# Check how strongly the three items are related.
scores = data[['c1', 'c2', 'c3']].astype(float)
display(scores.corr().round(3))
alpha = 3 / 2 * (1 - scores.var().sum() / scores.sum(axis=1).var())
print('Cronbach alpha:', round(alpha, 3))

## 4. Add age groups

18–24 is the reference group. Each other age group gets a column: 1 means the person is in that group, and 0 means they are not. There are no respondents aged 55–64, so that category is not estimated. This preserves age categories rather than inventing exact ages or equal distances.

**H0:** All five age-group paths to latent Control are zero. Individual path p-values are not a joint test of this hypothesis. These estimates are exploratory and are not used to select a significant age pair.

In [ ]:
# Make a 0/1 column for each age group.
age_groups = ['Under 18', '25-34', '35-44', '45-54', '65 and older']
assert data['age'].isin(['18-24'] + age_groups).all()
age = pd.get_dummies(data['age'])[age_groups]
age.columns = ['under18', 'age25', 'age35', 'age45', 'age65']
sem_data = pd.concat([scores, age], axis=1).astype(float)
display(data['age'].value_counts().to_frame('n'))

## 5. Define and fit the SEM

`=~` defines the measurement model. `~` defines regression paths. The first loading is set to 1. This gives the control factor a scale. We do not add extra links between item errors after seeing the results.

This uses the course example’s default **MLW** estimator and treats the five response categories as approximately continuous. Robust standard errors are requested below. They do not remove the equal-spacing assumption. Semopy also supports [ordinal specifications](https://semopy.com/ordinal.html); ordinal estimation is not implemented in this example.

The model assumes independent responses, one control factor, uncorrelated item residuals and no direct age-to-item paths (measurement invariance). These assumptions are not established by successful convergence. The small older groups cannot support a reliable full multi-group invariance study here.

In [ ]:
# Define the control factor and its link with age.
description = """
Control =~ c1 + c2 + c3
Control ~ under18 + age25 + age35 + age45 + age65
"""

# Fit the model, following the course example.
model = semopy.Model(description)
fit = model.fit(sem_data)
print(fit)
assert fit.success, 'The model did not converge.' 

## 6. Inspect the model

Loadings show how strongly each item relates to Control. Age-path estimates compare each group with 18–24 in latent-factor units, not directly in 1–5 points. The p-values are exploratory, uncorrected individual tests.

Inspect convergence, residual variances, standardized loadings and fit indices together. CFI/TLI can exceed 1 in the raw semopy output; such values do not demonstrate “better than perfect” validity. Fit indices for a model containing fixed observed age indicators need care and do not establish measurement quality.

In [ ]:
# Show standardised estimates and robust standard errors.
estimates = model.inspect(std_est=True, se_robust=True)
display(estimates.round(3))

# Show model fit measures, as in the course notebook.
fit_stats = semopy.calc_stats(model)
display(fit_stats[['DoF', 'chi2', 'chi2 p-value', 'CFI', 'TLI', 'RMSEA']].T.round(3))

# Check for negative variances. These would be a warning.
variances = estimates[(estimates['op'] == '~~') & (estimates['lval'] == estimates['rval'])]
print('Negative variances:', int((variances['Estimate'] < 0).sum()))

## What can we conclude?

The model converges for **671 respondents**, with no negative variances. However, the candidate scale has **Cronbach’s alpha ≈ 0.483**. The reversed algorithm item has a standardized loading of approximately **0.230**, compared with approximately **0.65** for the other two items. It contributes little to this candidate factor.

Therefore, this implementation demonstrates how to use semopy, but does **not validate a strong three-item control scale**. Do not interpret the age paths as a definitive answer to SQ2. Retain the original single-item SQ2 analysis, explain the weak measurement result and discuss the indicator selection with the lecturer. Do not automatically delete the weak item: a two-indicator factor needs additional identification assumptions.

No causal conclusion or full measurement-invariance conclusion follows from this cross-sectional model. The three-item factor is also a different measurement from the original single control item, so its coefficients cannot be directly equated with the earlier regression coefficients.

### Simple explanation for the defence

- **What is a latent factor?** A concept we cannot measure directly, such as control.
- **What does `=~` mean?** Several survey items measure one factor.
- **What does `~` mean?** A regression link between variables.
- **Why reverse c3?** A high original score means more control by the algorithm. Reversing it makes the direction match the other items.
- **Is the model good just because it runs?** No. The items are not strongly related.
- **What is the conclusion?** SEM was fitted, but the control scale is weak. This limits what we can conclude.

**Sources:** Course `SEM.ipynb` (political democracy example), [semopy tutorial](https://semopy.com/tutorial.html), [fit indices](https://semopy.com/indices.html). Data: `Responses (3).xlsx`, first worksheet. AI-generated code: review and attribute before submission.
